# RefCOCO 200-sample benchmark from Hugging Face

Downloads `lmms-lab-encoder/RefCOCO`, samples 200 validation examples, and exports images, referring expressions, bounding boxes, and GT masks for the MobileCLIP/MobileCLIP2/SigLIP semantic-ranking benchmark.

The HF dataset exposes `image`, `question`, `segmentation`, `bbox`, `iscrowd`, and `file_name`; its validation split has 8,811 examples. citeturn2search0

In [ ]:
!pip install -q datasets pycocotools pillow pandas matplotlib tqdm

In [ ]:
from pathlib import Path
import json, random, ast, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
N_SAMPLES = 200
HF_DATASET = "lmms-lab-encoder/RefCOCO"
HF_SPLIT = "val"

OUT = Path("/kaggle/working/refcoco_hf_benchmark_200")
IMG_OUT, MASK_OUT, DATA_OUT = OUT/"images", OUT/"masks", OUT/"data"
for p in [IMG_OUT, MASK_OUT, DATA_OUT]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
print("Output:", OUT)

In [ ]:
from datasets import load_dataset

ds = load_dataset(HF_DATASET, split=HF_SPLIT)
print(ds)
print("Columns:", ds.column_names)
print("Rows:", len(ds))

# Inspect one record before sampling
x = ds[0]
for k, v in x.items():
    print(k, type(v).__name__, getattr(v, "size", ""), str(v)[:300])

In [ ]:
# Reproducible 200-example subset.
# Prefer one referring expression per image so a single image does not dominate the benchmark.

shuffled = ds.shuffle(seed=SEED)
chosen = []
used = set()

for i in tqdm(range(len(shuffled)), desc="Sampling"):
    r = shuffled[i]
    key = str(r.get("file_name", i))
    if key in used:
        continue
    chosen.append(i)
    used.add(key)
    if len(chosen) == N_SAMPLES:
        break

if len(chosen) < N_SAMPLES:
    raise RuntimeError(f"Only found {len(chosen)} unique-image examples.")

bench = shuffled.select(chosen)
print("Rows:", len(bench), "Unique images:", len(set(bench["file_name"])))

In [ ]:
from pycocotools import mask as mask_utils

def parse_bbox(x):
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except Exception:
            x = json.loads(x)
    return [float(v) for v in x]

def decode_segmentation(seg, w, h):
    # The HF schema stores segmentation as a float sequence.
    # Current RefCOCO conversion represents it as polygon coordinates.
    if isinstance(seg, np.ndarray):
        seg = seg.tolist()

    if not isinstance(seg, list) or not seg:
        raise ValueError(f"Unsupported segmentation: {type(seg)}")

    if isinstance(seg[0], (list, tuple, np.ndarray)):
        polys = [[float(v) for v in p] for p in seg]
    else:
        polys = [[float(v) for v in seg]]

    polys = [p for p in polys if len(p) >= 6 and len(p) % 2 == 0]
    if not polys:
        raise ValueError(f"Could not decode polygon: {str(seg)[:300]}")

    rles = mask_utils.frPyObjects(polys, int(h), int(w))
    rle = mask_utils.merge(rles if isinstance(rles, list) else [rles])
    m = mask_utils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2)
    return m.astype(np.uint8)

In [ ]:
# Export frozen benchmark
rows = []

for i in tqdm(range(len(bench)), desc="Exporting"):
    r = bench[i]
    image = r["image"].convert("RGB")
    w, h = image.size

    uid = f"{i:04d}_{r['question_id']}"
    image_name = uid + ".jpg"
    mask_name = uid + ".png"

    image.save(IMG_OUT / image_name, quality=95)

    mask = decode_segmentation(r["segmentation"], w, h)
    Image.fromarray(mask * 255).save(MASK_OUT / mask_name)

    rows.append({
        "local_id": i,
        "question_id": str(r["question_id"]),
        "file_name": str(r["file_name"]),
        "image_file": f"images/{image_name}",
        "mask_file": f"masks/{mask_name}",
        "question": str(r["question"]),
        "bbox": parse_bbox(r["bbox"]),
        "image_width": w,
        "image_height": h,
        "iscrowd": int(r["iscrowd"]),
    })

df = pd.DataFrame(rows)
assert len(df) == N_SAMPLES
assert df.question_id.is_unique

df.to_csv(OUT/"benchmark_200.csv", index=False)
df.to_parquet(DATA_OUT/"benchmark_200.parquet", index=False)

manifest = {
    "source": HF_DATASET,
    "split": HF_SPLIT,
    "samples": N_SAMPLES,
    "seed": SEED,
    "sampling": "one referring expression per image",
    "models": ["MobileCLIP-S0","MobileCLIP-S2","MobileCLIP2-S0","MobileCLIP2-S2","SigLIP-B/16"]
}
with open(OUT/"MANIFEST.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(df.head())
print("Saved:", OUT/"benchmark_200.csv")

In [ ]:
# Integrity checks
for _, r in tqdm(df.iterrows(), total=len(df), desc="Checking"):
    im = np.asarray(Image.open(OUT/r.image_file))
    m = np.asarray(Image.open(OUT/r.mask_file))
    assert im.shape[:2] == m.shape[:2]
    assert set(np.unique(m)).issubset({0,255})
    assert (m > 0).any()

print("✓ All 200 images have matching binary GT masks")

In [ ]:
# Preview
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for ax, (_, r) in zip(axes.ravel(), df.head(12).iterrows()):
    im = np.asarray(Image.open(OUT/r.image_file).convert("RGB"))
    m = np.asarray(Image.open(OUT/r.mask_file)) > 0
    ax.imshow(im)
    ax.imshow(np.ma.masked_where(~m, m), alpha=.45)
    ax.set_title(r.question[:70])
    ax.axis("off")
for ax in axes.ravel()[len(df.head(12)):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT/"preview_examples.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Statistics
df["question_len"] = df.question.str.split().str.len()
print("Samples:", len(df))
print("Unique images:", df.file_name.nunique())
display(df[["question_len"]].describe())
display(df[["question_id","question","bbox"]].head(20))

# ZIP
zip_path = shutil.make_archive(
    "/kaggle/working/refcoco_hf_benchmark_200",
    "zip",
    root_dir="/kaggle/working",
    base_dir="refcoco_hf_benchmark_200"
)
print("ZIP:", zip_path)

## Downstream evaluation

For each frozen sample:

`image + referring expression → candidate masks → image-text similarity → top-1 mask → IoU with GT`

Report:

- Top-1 selection accuracy
- Top-5 selection accuracy
- mean/median IoU
- Recall@IoU ≥ 0.5
- CPU latency
- model size

Keep this exact 200-sample manifest fixed across MobileCLIP-S0/S2, MobileCLIP2-S0/S2, and SigLIP-B/16.